<a href="https://colab.research.google.com/github/KRSets07/python-course-project/blob/main/week_7_aassignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Selected Pretrained Model

**Model:** TabPFN v2 Classifier

**Hugging Face link:** https://huggingface.co/Prior-Labs/TabPFN-v2-clf

TabPFN is a pretrained, transformer-based model designed for classification using structured tabular data. It can make predictions on small datasets without requiring the user to train a new model from scratch.

## 2. What Does TabPFN v2 Do?

TabPFN v2 is a pretrained foundation model designed to solve classification problems involving structured tabular data.

It uses a transformer architecture and has been pretrained on a large collection of synthetic datasets. During use, it receives a labelled training dataset and new, unseen observations. It uses patterns learned during pretraining, together with the examples supplied by the user, to predict the class of the unseen observations.

**Input:**
- A labelled tabular dataset containing predictor variables and their known classes.
- New, unseen observations with the same predictor variables.

**Output:**
- Predicted class labels for the unseen observations.
- Class probabilities indicating the model's confidence in each prediction.

Unlike conventional machine learning models, TabPFN does not need to be trained from scratch on every new dataset. It uses its pretrained knowledge to make predictions from the examples provided.

## 3. Relevance to Geotechnical Engineering and My Research

TabPFN v2 could be useful in geotechnical engineering, particularly where datasets are small and contain several interacting variables.

In my research on open-stope stability, I use geological, geotechnical and geometrical measurements to investigate the stability of hangingwall and footwall surfaces. TabPFN could be evaluated using these measurements to classify stope surfaces as stable or unstable based on observed ELOS values.

Its ability to work with small tabular datasets makes it relevant to mining environments where collecting large amounts of reliable geotechnical data can be difficult.

Comparing its predictions with my existing Extra Trees classifier would also help establish whether a pretrained model offers any improvement in stability prediction. This would need to be tested using independent data and appropriate validation methods.

## 4. Comparison with My Existing Model

My research uses an Extra Trees classifier, which learns patterns by building an ensemble of decision trees using mine-specific training data.

TabPFN v2 uses a different approach. It is a pretrained transformer model that has already learned patterns from many synthetic tabular datasets. It uses this pretrained knowledge, together with the labelled examples supplied during use, to predict outcomes for new observations.

The main differences are:

- **Extra Trees:** Learns patterns directly from the mine-specific training dataset.
- **TabPFN v2:** Uses knowledge gained during pretraining and applies it to the supplied dataset.
- **Extra Trees:** Builds multiple decision trees to make predictions.
- **TabPFN v2:** Uses a transformer architecture to make predictions.

Both models can perform classification, but they use different learning approaches.

## 5. Potential Applications of Pretrained Models, Generative AI and NLP

Pretrained models, generative AI and Natural Language Processing (NLP) offer different ways to support geotechnical engineering research.

- **Pretrained models:** Models such as TabPFN could be used to analyse structured geotechnical datasets and support the prediction of outcomes such as stope stability.
- **Generative AI:** Could assist with summarising research papers, explaining technical concepts and supporting the preparation of research reports.
- **Natural Language Processing (NLP):** Could extract useful information from geological reports, core logging records and other technical documents.

Together, these technologies could help researchers analyse data, organise technical information and improve research efficiency. However, their outputs must be checked against reliable data and engineering knowledge.

## 6. Limitations

Despite its advantages, TabPFN v2 has several limitations:

- **Data quality:** Poor-quality or incorrect input data can affect prediction accuracy.
- **Generalisation:** Patterns learned from synthetic datasets may not fully represent real geological and geotechnical conditions.
- **Interpretability:** Understanding why the model makes a particular prediction can be difficult.
- **Computational requirements:** Running the model may require suitable computing resources, particularly for larger datasets.
- **Validation:** Its performance must be tested on independent data before its predictions can be trusted in engineering applications.

For mining applications, model predictions should support, rather than replace, engineering judgement and site-specific assessments.

In [ ]:
from __future__ import annotations
import argparse
import hashlib
import itertools
import json
import platform
import sys
from io import BytesIO
from inspect import signature
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy.interpolate import PchipInterpolator
from scipy.optimize import minimize
from scipy.special import expit, logit
from scipy.spatial import Delaunay
from scipy.stats import binomtest
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (ExtraTreesClassifier, RandomForestClassifier,
                              ExtraTreesRegressor, RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, precision_score, recall_score, mean_absolute_error,
    mean_squared_error, r2_score, roc_curve, precision_recall_curve)

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred, *, sample_weight=None,
                                multioutput='uniform_average'):
        errors = np.sqrt(mean_squared_error(
            y_true, y_pred, sample_weight=sample_weight, multioutput='raw_values'))
        if isinstance(multioutput, str):
            if multioutput == 'raw_values':
                return errors
            if multioutput != 'uniform_average':
                raise ValueError("multioutput must be 'raw_values', 'uniform_average', or output weights")
            multioutput = None
        return float(np.average(errors, weights=multioutput))

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, RandomizedSearchCV
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Arc

SEED = 42
BOOTSTRAPS = 2000
BASE_NUM = ['Q_prime','B','C','HR','N_prime','Top_Length_m','Bottom_Length_m',
            'North_Height_m','South_Height_m']
LOG_NUM = [x for x in BASE_NUM if x != 'B']
RENAME = {"Q'":'Q_prime',"N'":'N_prime','ELOS (m)':'ELOS_m',
 'Top Length (m)':'Top_Length_m','Bottom Length (m)':'Bottom_Length_m',
 'North Height (m)':'North_Height_m','South Height (m)':'South_Height_m'}
METRICS = ['Accuracy','Balanced_accuracy','F1','MCC','ROC_AUC','AP','Brier']
COLOURS = ['#25476A','#2A9D8F','#E76F51','#756BB1','#B8860B']

def emit(message):
    print(message, flush=True)

def features(raw, kind='full', surface=True, interaction=True):

    if kind == 'simple':
        x = pd.DataFrame({'log_HR':np.log(raw.HR), 'log_N_prime':np.log(raw.N_prime)},index=raw.index)
    else:
        x = raw[BASE_NUM].copy()
        if kind == 'full':
            for c in LOG_NUM:
                x['log1p_'+c] = np.log1p(raw[c])
            x['Top_over_Bottom'] = raw.Top_Length_m/raw.Bottom_Length_m
            x['Height_ratio'] = raw.North_Height_m/raw.South_Height_m
            x['Perimeter_proxy'] = raw[['Top_Length_m','Bottom_Length_m',
                                      'North_Height_m','South_Height_m']].sum(axis=1)
            if interaction:
                x['HR_x_N_prime'] = raw.HR*raw.N_prime
    if surface:
        x['Surface'] = raw.Surface
    return x

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, kind='full', surface=True, interaction=True):
        self.kind, self.surface, self.interaction = kind, surface, interaction
    def fit(self, X, y=None):
        self.feature_names_out_ = np.asarray(features(X,self.kind,self.surface,self.interaction).columns)
        return self
    def transform(self, X):
        return features(X,self.kind,self.surface,self.interaction)
    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

def dense_one_hot_encoder():

    options = {'drop': 'if_binary', 'handle_unknown': 'ignore'}

    dense_option = 'sparse_output' if 'sparse_output' in signature(OneHotEncoder).parameters else 'sparse'
    options[dense_option] = False
    return OneHotEncoder(**options)

def pipeline(model, sample, kind='full', surface=True, interaction=True):
    names = features(sample,kind,surface,interaction).columns
    nums = [c for c in names if c != 'Surface']
    transforms = [('num',Pipeline([('impute',SimpleImputer(strategy='median')),
                                   ('scale',StandardScaler())]),nums)]
    if surface:
        transforms.append(('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
            ('encode',dense_one_hot_encoder())]),['Surface']))
    return Pipeline([('features',FeatureBuilder(kind,surface,interaction)),
                     ('preprocess',ColumnTransformer(transforms,verbose_feature_names_out=False)),
                     ('model',model)])


In [ ]:
def splits(X,y,g, n=5, seed=SEED):
    ss=list(StratifiedGroupKFold(n_splits=n,shuffle=True,random_state=seed).split(X,y,g))
    for a,b in ss:
        assert not set(g.iloc[a]) & set(g.iloc[b])
        assert len(np.unique(y[a]))==2 and len(np.unique(y[b]))==2
    return ss

SEARCH = {'model__n_estimators':[300,600], 'model__max_depth':[8,12,None],
          'model__min_samples_split':[2,6], 'model__min_samples_leaf':[1,2,4],
          'model__max_features':['sqrt',1.0], 'model__bootstrap':[False,True]}

def search_fit(template,X,y,g):
    cv=splits(X,y,g,n=3)
    search=RandomizedSearchCV(clone(template),SEARCH,n_iter=16,cv=cv,
        scoring='balanced_accuracy',random_state=SEED,n_jobs=4,refit=True,error_score='raise')
    search.fit(X,y)
    return search.best_estimator_,search.best_params_,float(search.best_score_)

def raw_oof(template,X,y,g,cv,tune=False):
    p=np.full(len(y),np.nan); fold_rows=[]; params=[]
    for i,(a,b) in enumerate(cv):
        if tune:
            fitted,settings,score=search_fit(template,X.iloc[a],y[a],g.iloc[a])
            params.append({'fold':i+1,'inner_score':score,'parameters':settings})
        else:
            fitted=clone(template).fit(X.iloc[a],y[a])
        p[b]=fitted.predict_proba(X.iloc[b])[:,1]
        fold_rows.append({'fold':i+1,**metric_values(y[b],p[b],0.5)})
    assert np.isfinite(p).all()
    return p,fold_rows,params

def sigmoid_fit(y,p,positive_slope=True):
    z=logit(np.clip(p,1e-6,1-1e-6)); D=np.column_stack([np.ones(len(y)),z])
    def objective(theta):
        eta=D@theta
        return np.logaddexp(0,eta).sum()-np.dot(y,eta)+1e-8*np.dot(theta,theta)
    def gradient(theta):
        return D.T@(expit(D@theta)-y)+2e-8*theta
    bounds=[(-50,50),(0,50) if positive_slope else (-50,50)]
    r=minimize(objective,[0.,1.],jac=gradient,method='L-BFGS-B',bounds=bounds,
               options={'maxiter':1000,'ftol':1e-12,'gtol':1e-8})
    if not r.success and np.linalg.norm(gradient(r.x))>1e-4:
        raise RuntimeError('Sigmoid fit failed: '+r.message)
    return r.x

def sigmoid_apply(p,theta):
    return expit(theta[0]+theta[1]*logit(np.clip(p,1e-6,1-1e-6)))

def cross_calibration(y,raw_p,cv):
    p=np.full(len(y),np.nan)
    for a,b in cv:
        p[b]=sigmoid_apply(raw_p[b],sigmoid_fit(y[a],raw_p[a]))
    return p, sigmoid_fit(y,raw_p)

def choose_threshold(y,p):
    rows=[]
    for threshold in np.linspace(.05,.95,181):
        pred=p>=threshold
        rows.append({'threshold':float(threshold),
            'Balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'F1':float(f1_score(y,pred,zero_division=0)),
            'Recall':float(recall_score(y,pred,zero_division=0)),
            'Accuracy':float(accuracy_score(y,pred)), 'distance_from_half':abs(threshold-.5)})
    frame=pd.DataFrame(rows)
    best=frame.sort_values(['Balanced_accuracy','distance_from_half','Recall'],
                          ascending=[False,True,False],kind='stable').iloc[0]
    return float(best.threshold),frame

def metric_values(y,p,threshold=.5):
    pred=(np.asarray(p)>=threshold).astype(int)
    two=len(np.unique(y))==2
    return {'Accuracy':float(accuracy_score(y,pred)),
        'Balanced_accuracy':float(balanced_accuracy_score(y,pred)) if two else np.nan,
        'F1':float(f1_score(y,pred,zero_division=0)),
        'MCC':float(matthews_corrcoef(y,pred)),
        'ROC_AUC':float(roc_auc_score(y,p)) if two else np.nan,
        'AP':float(average_precision_score(y,p)) if two else np.nan,
        'Brier':float(brier_score_loss(y,p)),
        'Precision':float(precision_score(y,pred,zero_division=0)),
        'Recall':float(recall_score(y,pred,zero_division=0))}

def cluster_indices(groups,n=BOOTSTRAPS,seed=SEED):
    unique=pd.unique(groups); locations=[np.flatnonzero(np.asarray(groups)==v) for v in unique]
    rng=np.random.default_rng(seed)
    return [np.concatenate([locations[j] for j in rng.integers(0,len(unique),len(unique))]) for _ in range(n)]

def metric_intervals(y,p,threshold,resamples):
    result=pd.DataFrame([metric_values(y[ix],p[ix],threshold) for ix in resamples])
    return {key:(float(result[key].quantile(.025)),float(result[key].quantile(.975))) for key in result}

def calibration_diagnostics(y,p):
    theta=sigmoid_fit(y,p,positive_slope=False)
    z=logit(np.clip(p,1e-6,1-1e-6))
    offset=minimize(lambda a:np.logaddexp(0,a[0]+z).sum()-np.dot(y,a[0]+z),
                    [0.],method='BFGS').x[0]
    return {'Calibration_intercept':float(offset),'Calibration_joint_intercept':float(theta[0]),
            'Calibration_slope':float(theta[1]),'Brier':float(brier_score_loss(y,p))}

def paired_compare(y,p1,t1,p2,t2,g,resamples):
    c1=(p1>=t1)==y; c2=(p2>=t2)==y
    b=int(np.sum(c1 & ~c2)); c=int(np.sum(~c1 & c2))
    differences=np.array([(c1-c2.astype(int))[np.asarray(g)==s].sum() for s in pd.unique(g)])
    rng=np.random.default_rng(SEED); signs=rng.choice([-1,1],size=(100000,len(differences)))
    stat=abs(differences.sum())
    pv=(1+np.sum(np.abs(signs@differences)>=stat))/(100001)
    delta=np.array([np.mean(c1[ix])-np.mean(c2[ix]) for ix in resamples])
    return {'Accuracy_difference':float(np.mean(c1)-np.mean(c2)),
            'Difference_CI_low':float(np.quantile(delta,.025)),
            'Difference_CI_high':float(np.quantile(delta,.975)),
            'Model1_only_correct':b,'Model2_only_correct':c,
            'Exact_row_McNemar_p':float(binomtest(b,b+c,.5).pvalue) if b+c else 1.,
            'Stope_cluster_permutation_p':float(pv)}


In [ ]:
def boundary_load(path):
    result={}
    for name,frame in pd.read_excel(path,sheet_name=None).items():
        frame.columns=[str(c).strip() for c in frame.columns]
        frame=frame.iloc[:,:2].copy();frame.columns=['HR','N_prime']
        frame=frame.dropna().sort_values('HR')
        if frame.HR.duplicated().any():
            raise ValueError('Duplicate hydraulic-radius coordinates in '+name)
        assert (frame.N_prime>0).all() and np.all(np.diff(frame.N_prime)>0)
        result[name]=frame
    return result

def boundary_prediction(raw,boundaries,name='Stable'):
    b=boundaries[name]
    fun=PchipInterpolator(b.HR,np.log(b.N_prime),extrapolate=False)
    threshold=np.exp(fun(raw.HR))
    if not np.isfinite(threshold).all():
        raise ValueError('Empirical baseline queried beyond digitised boundary support')
    return (raw.N_prime.to_numpy()<threshold).astype(int),threshold

def save_figure(fig,folder,name):
    for extension in ['pdf','svg','png']:
        target=folder/(name+'.'+extension)
        temporary=folder/('.'+name+'.'+extension+'.tmp')
        buffer=BytesIO()
        fig.savefig(buffer,format=extension,dpi=600,bbox_inches='tight')
        exported=buffer.getvalue()
        if len(exported)<100 or (extension=='pdf' and not exported.startswith(b'%PDF-')):
            raise IOError('Invalid figure export: '+name+'.'+extension)
        temporary.write_bytes(exported)
        if temporary.read_bytes()!=exported:
            raise IOError('Incomplete figure write: '+name+'.'+extension)
        temporary.replace(target)
    plt.close(fig)

def mathews_axes(ax):
    ax.set(xlabel='Hydraulic radius HR (m)',ylabel="Modified stability number $N'$",yscale='log')
    ax.grid(True,alpha=.16,which='both')

def plot_boundaries(ax,boundaries,alpha=.6):
    for i,(name,b) in enumerate(boundaries.items()):
        ax.plot(b.HR,b.N_prime,color='.35',linestyle=['-','--','-.',':'][i],
                lw=.9,alpha=alpha,label='Potvin '+name)

def supported_projection(raw,p,grid_hr,grid_n,k=5):

    coordinates=np.column_stack([raw.HR,np.log(raw.N_prime)])
    center=coordinates.mean(axis=0); scale=coordinates.std(axis=0,ddof=1)
    z=(coordinates-center)/scale
    radius=[]
    for j in range(len(raw)):
        candidates=z[raw.stope_id.to_numpy()!=raw.stope_id.iloc[j]]
        ds=np.sort(np.linalg.norm(candidates-z[j],axis=1))
        radius.append(ds[k-1])
    cut=float(np.quantile(radius,.95))
    a,b=np.meshgrid(grid_hr,grid_n)
    q=(np.column_stack([a.ravel(),np.log(b.ravel())])-center)/scale
    distances,ix=NearestNeighbors(n_neighbors=k).fit(z).kneighbors(q)
    hull=Delaunay(z).find_simplex(q)>=0
    supported=hull & (distances[:,-1]<=cut)
    w=np.exp(-.5*(distances/np.maximum(distances[:,-1,None],1e-9))**2)
    value=(w*np.asarray(p)[ix]).sum(axis=1)/w.sum(axis=1)
    value[~supported]=np.nan
    return value.reshape(a.shape),supported.reshape(a.shape),cut,distances[:,-1].reshape(a.shape)

def postchecks(out):

    out=Path(out); tables=out/'tables'
    summary=json.loads((out/'analysis_summary.json').read_text())
    pred=pd.read_csv(tables/'holdout_predictions.csv')
    results=pd.read_csv(tables/'holdout_results.csv')
    y=pred.observed_unstable.to_numpy(); selected=summary['selected_model']
    for _,row in results.iterrows():
        yy=pred[row.Model+'_predicted'].to_numpy()
        assert np.isclose(accuracy_score(y,yy),row.Accuracy)
        assert np.isclose(balanced_accuracy_score(y,yy),row.Balanced_accuracy)
        assert np.isclose(f1_score(y,yy),row.F1)
    split=pd.read_csv(tables/'split_manifest.csv')
    a=set(split.loc[split.subset=='development','stope_id'])
    b=set(split.loc[split.subset=='holdout','stope_id'])
    assert not a & b and len(a)==108 and len(b)==37

    calibrated=pred[selected+'_probability'].to_numpy()
    intercept,slope=summary['calibrators'][selected]
    raw=expit((logit(calibrated)-intercept)/slope)
    resamples=cluster_indices(pred.stope_id,n=500,seed=91)
    rows=[]
    for stage,p in [('raw',raw),('calibrated',calibrated)]:
        values=[calibration_diagnostics(y[ix],p[ix]) for ix in resamples if len(np.unique(y[ix]))==2]
        vals=calibration_diagnostics(y,p)
        for key in ['Calibration_intercept','Calibration_slope']:
            vv=np.array([r[key] for r in values]); vals[key+'_CI_low'],vals[key+'_CI_high']=np.quantile(vv,[.025,.975])
        rows.append({'Model':selected,'Stage':stage,**vals})
    pd.DataFrame(rows).to_csv(tables/'selected_calibration_cluster_intervals.csv',index=False)
    audit=pd.read_csv(tables/'source_record_audit.csv');valid=audit[audit.analysis_status=='Included']
    paired=valid.assign(unstable=valid.ELOS_m>=.6).pivot(index='stope_id',columns='Surface',values='unstable')
    hw_only=int((paired.HW & ~paired.FW).sum());fw_only=int((~paired.HW & paired.FW).sum())
    observed={'HW_unstable':int(paired.HW.sum()),'FW_unstable':int(paired.FW.sum()),
        'HW_only_unstable':hw_only,'FW_only_unstable':fw_only,
        'Paired_surface_exact_McNemar_p':float(binomtest(hw_only,hw_only+fw_only,.5).pvalue)}
    grid=pd.read_csv(tables/'supported_probability_grid.csv')
    support={s:float(grid[s+'_supported'].mean()) for s in ['HW','FW']}
    support['both']=float((grid.HW_supported & grid.FW_supported).mean())
    check={'status':'passed','test_records':len(pred),'test_stopes':len(b),
           'no_stope_overlap':True,'all_holdout_metrics_reconciled':True,
           'observed_paired_surface_association':observed,'grid_support_fractions':support}
    (out/'verification.json').write_text(json.dumps(check,indent=2))
    emit('Numerical reconciliation passed; calibration cluster intervals exported.')
    return check


In [ ]:
def outside_legend(fig, axes, ncol=3, fontsize=8, bottom=0.10):
    entries = {}
    for ax in np.asarray(axes, dtype=object).reshape(-1):
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels):
            if label and not label.startswith('_'):
                entries.setdefault(label, handle)
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    fig.tight_layout(rect=(0, bottom, 1, 1), pad=1.0, w_pad=1.8)
    return fig.legend(list(entries.values()), list(entries), loc='lower center',
                      bbox_to_anchor=(0.5, 0.015), ncol=ncol, frameon=False,
                      fontsize=fontsize, handlelength=2.6, handletextpad=0.7,
                      columnspacing=1.7, labelspacing=0.55)

def analysis_pipeline_figure():
    fig, ax = plt.subplots(figsize=(7.2, 6.6))
    fig.subplots_adjust(left=0.025, right=0.975, bottom=0.02, top=0.98)
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.axis('off')
    navy, teal, amber = '#25476A', '#238F86', '#A97820'
    ink, muted, edge = '#202D3B', '#536475', '#D6DFE7'

    def card(x, y, width, height, number, heading, accent=navy, face='white'):
        ax.add_patch(FancyBboxPatch((x, y), width, height,
                     boxstyle='round,pad=0.006,rounding_size=0.012',
                     linewidth=0.9, edgecolor=edge, facecolor=face, zorder=2))
        ax.add_patch(FancyBboxPatch((x + 0.014, y + height - 0.047),
                     0.038, 0.030, boxstyle='round,pad=0.002,rounding_size=0.005',
                     linewidth=0, facecolor=accent, zorder=3))
        ax.text(x + 0.033, y + height - 0.032, number, ha='center', va='center',
                fontsize=8.5, color='white', weight='bold', zorder=4)
        ax.text(x + 0.065, y + height - 0.032, heading,
                fontsize=9.4, weight='bold', color=accent, va='center', zorder=4)

    def text(x, y, wording, size=9.5, bold=False, color=ink, align='left'):
        ax.text(x, y, wording, fontsize=size, color=color,
                weight='bold' if bold else 'normal', va='center', ha=align,
                linespacing=1.55, zorder=4)

    def arrow(start, end, color=navy, style='-'):
        ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>',
                     mutation_scale=11, linewidth=1.15, color=color,
                     linestyle=style, shrinkA=0, shrinkB=0, zorder=1))

    card(0.05, 0.855, 0.90, 0.118, '01', 'INPUT AUDIT')
    text(0.077, 0.910, '294 surface records  /  147 paired stopes', 13, True)
    text(0.077, 0.876, 'Check stope identifiers, paired surfaces and stability inputs', 9.5, color=muted)

    card(0.05, 0.700, 0.90, 0.118, '02', 'COMMON ANALYSIS COHORT', teal, '#F3F9F8')
    text(0.077, 0.755, '290 valid records  /  145 complete pairs', 13, True)
    text(0.077, 0.721, "Four records excluded: missing $Q'$ and zero $N'$", 9.5, color=muted)
    arrow((0.5, 0.849), (0.5, 0.824))

    ax.plot([0.27, 0.73], [0.672, 0.672], color=navy, lw=1.15, zorder=1)
    ax.plot([0.5, 0.5], [0.694, 0.672], color=navy, lw=1.15, zorder=1)
    arrow((0.27, 0.672), (0.27, 0.641), teal)
    arrow((0.73, 0.672), (0.73, 0.641), amber)
    text(0.5, 0.653, 'Stope-level split', 8.5, color=muted, align='center')

    card(0.05, 0.507, 0.42, 0.128, '03', 'DEVELOPMENT', teal, '#F3F9F8')
    text(0.076, 0.566, '108 stopes  /  216 records', 11.7, True)
    text(0.076, 0.530, 'Model choices use development data', 9.5, color=muted)

    card(0.55, 0.453, 0.40, 0.182, '04', 'RESERVED HOLDOUT', amber, '#FBF8F1')
    text(0.576, 0.565, '37 stopes  /  74 records', 11.7, True)
    text(0.576, 0.519, 'Distinct stope identifiers\nHeld until settings are frozen', 9.5, color=muted)
    ax.add_patch(Arc((0.910, 0.607), 0.020, 0.022, theta1=0, theta2=180,
                     linewidth=1.1, color=amber, zorder=4))
    ax.add_patch(Rectangle((0.897, 0.586), 0.026, 0.022,
                     linewidth=1.1, edgecolor=amber, facecolor='none', zorder=4))

    card(0.05, 0.288, 0.42, 0.177, '05', 'MODEL DEVELOPMENT', teal)
    text(0.076, 0.400, 'Grouped CV and nested tree search', 9.5, True)
    text(0.076, 0.351, 'OOF sigmoid calibration\nDevelopment threshold selection', 9.5, color=muted)
    arrow((0.26, 0.501), (0.26, 0.471), teal)

    card(0.55, 0.288, 0.40, 0.122, '06', 'MATCHED ASSESSMENT', navy)
    text(0.576, 0.341, 'Evaluate the same 74 records', 9.6, True)
    text(0.576, 0.310, 'Chart, logistic and ensemble models', 9.5, color=muted)
    arrow((0.75, 0.447), (0.75, 0.416), amber)

    ax.add_patch(FancyBboxPatch((0.480, 0.353), 0.060, 0.051,
                 boxstyle='round,pad=0.003,rounding_size=0.006',
                 linewidth=0, facecolor='#E8EEF4', zorder=2))
    text(0.510, 0.379, 'Freeze', 8.5, True, navy, 'center')
    arrow((0.476, 0.340), (0.544, 0.340))

    card(0.05, 0.059, 0.42, 0.180, '07', 'SUPPORTED PROJECTIONS', teal, '#F3F9F8')
    text(0.076, 0.174, 'Calibrated OOF record probabilities', 9.5, True)
    text(0.076, 0.122, "HW and FW maps in HR-$N'$ space\nMask insufficient local support", 9.5, color=muted)
    arrow((0.26, 0.282), (0.26, 0.245), teal)

    card(0.55, 0.059, 0.40, 0.180, '08', 'ROBUSTNESS CHECKS', navy)
    text(0.576, 0.136, 'Surface / interaction ablations\nELOS sensitivity and regression\nCalibration; stope bootstrap intervals',
         9.5, color=muted)
    arrow((0.75, 0.282), (0.75, 0.245))
    return fig

def make_figures(df,train,test,ytrain,ytest,models,selection,cvframe,foldframe,
                 preds,thresholds,curves,perm,importance,boundaries,maps,grids,
                 empirical,ablation,sensitivity,regression,out):
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':11,
        'axes.labelsize':10,'legend.fontsize':9,'pdf.fonttype':42,'ps.fonttype':42,
        'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':140,
        'svg.fonttype':'none'})
    figdir=out/'figures';figdir.mkdir(exist_ok=True)

    fig,ax=plt.subplots(figsize=(7,4.8));plot_boundaries(ax,boundaries,1)
    mathews_axes(ax);ax.set(xlim=(1,25),ylim=(.1,700));ax.legend(ncol=2,loc='upper left')
    save_figure(fig,figdir,'01_potvin_boundaries')
    fig,axes=plt.subplots(1,2,figsize=(8,4.25))
    for j,s in enumerate(['HW','FW']):
        axes[0].hist(df.loc[df.Surface==s,'ELOS_m'],bins=np.linspace(0,4,21),
                     histtype='step',lw=1.8,color=COLOURS[j],label=s)
    for t in [.5,.6,.8]: axes[0].axvline(t,c='.4',ls='--',lw=.9)
    axes[0].set(xlabel='Observed ELOS (m)',ylabel='Surface records')
    cts=[sum(df.ELOS_m<t) for t in [.5,.6,.8]]
    x=np.arange(3); axes[1].bar(x,cts,color=COLOURS[1],label='Stable')
    axes[1].bar(x,len(df)-np.asarray(cts),bottom=cts,color=COLOURS[2],label='Unstable')
    axes[1].set(xticks=x,xticklabels=['0.5','0.6','0.8'],xlabel='ELOS cutoff (m)',ylabel='Surface records')
    outside_legend(fig,axes,ncol=4,fontsize=9,bottom=0.07)
    save_figure(fig,figdir,'02_elos_threshold_sensitivity')
    save_figure(analysis_pipeline_figure(),figdir,'03_grouped_analysis_pipeline')
    fig,ax=plt.subplots(figsize=(5,3.6))
    counts=pd.Series((df.ELOS_m>=.6).astype(int)).value_counts().reindex([0,1])
    ax.bar(['Stable','Unstable'],counts,color=[COLOURS[1],COLOURS[2]])
    for i,v in enumerate(counts): ax.text(i,v+3,str(v),ha='center')
    ax.set(ylabel='Valid surface records',ylim=(0,230));save_figure(fig,figdir,'04_target_distribution')
    fig,ax=plt.subplots(figsize=(8,4));names=list(cvframe.Model);x=np.arange(len(names))
    for j,key in enumerate(['Accuracy','Balanced_accuracy','F1']):
        ax.bar(x+(j-1)*.23,cvframe[key+'_mean'],width=.23,
               yerr=cvframe[key+'_sd'],capsize=3,color=COLOURS[j],label=key.replace('_',' '))
    ax.set(ylim=(0,1),ylabel='Grouped development CV score',xticks=x,
           xticklabels=[n.replace('_','\n') for n in names]);ax.legend(ncol=3,loc='upper center')
    fig.tight_layout();save_figure(fig,figdir,'05_grouped_cv_comparison')
    p=preds[selection]['calibrated']; t=thresholds[selection]
    fig,ax=plt.subplots(figsize=(4.3,3.9));cm=confusion_matrix(ytest,p>=t)
    ax.imshow(cm,cmap='Blues')
    for i,j in itertools.product(range(2),repeat=2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=14)
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Stable','Unstable'],yticklabels=['Stable','Unstable'],
           xlabel='Predicted class',ylabel='Observed class');save_figure(fig,figdir,'06_holdout_confusion_matrix')
    fig,axes=plt.subplots(1,2,figsize=(8,3.7))
    for j,n in enumerate([selection,'Chart_logistic','Random_Forest']):
        pp=preds[n]['calibrated'];fpr,tpr,_=roc_curve(ytest,pp);precision,recall,_=precision_recall_curve(ytest,pp)
        axes[0].plot(fpr,tpr,color=COLOURS[j],label=n.replace('_',' ')+' '+f'{roc_auc_score(ytest,pp):.3f}')
        axes[1].plot(recall,precision,color=COLOURS[j],label=n.replace('_',' ')+' '+f'{average_precision_score(ytest,pp):.3f}')
    axes[0].plot([0,1],[0,1],'--',color='.6');axes[1].axhline(ytest.mean(),ls='--',c='.6')
    axes[0].set(xlabel='False positive rate',ylabel='True positive rate');axes[1].set(xlabel='Recall',ylabel='Precision')
    for a in axes: a.legend(fontsize=7);a.set(xlim=(0,1),ylim=(0,1.02))
    fig.tight_layout();save_figure(fig,figdir,'07_holdout_roc_precision_recall')
    fig,ax=plt.subplots(figsize=(8,7));f=features(train).drop(columns='Surface')
    corr=f.corr();im=ax.imshow(np.ma.masked_where(np.triu(np.ones(corr.shape),1),corr),cmap='coolwarm',vmin=-1,vmax=1)
    labels=[c.replace('log1p_','log ').replace('_m','').replace('_',' ') for c in corr.columns]
    ax.set(xticks=range(len(labels)),yticks=range(len(labels)),xticklabels=labels,yticklabels=labels)
    ax.tick_params(labelsize=7);plt.setp(ax.get_xticklabels(),rotation=90);fig.colorbar(im,ax=ax,shrink=.7,label='Pearson correlation')
    fig.tight_layout();save_figure(fig,figdir,'08_training_feature_correlation')
    fig,axes=plt.subplots(1,2,figsize=(8,4.4),sharex=True,sharey=True)
    for ax,s in zip(axes,['HW','FW']):
        sub=df[df.Surface==s]
        for cls,marker,col in [(0,'o',COLOURS[1]),(1,'^',COLOURS[2])]:
            q=sub[(sub.ELOS_m>=.6).astype(int)==cls]
            ax.scatter(q.HR,q.N_prime,s=20,marker=marker,c=col,alpha=.7,label=['Stable','Unstable'][cls])
        mathews_axes(ax);plot_boundaries(ax,boundaries);ax.set(title=s,xlim=(5.4,9.1),ylim=(2,55))
    outside_legend(fig,axes,ncol=3,fontsize=8,bottom=0.10)
    save_figure(fig,figdir,'09_observed_mathews_space')
    fig,ax=plt.subplots(figsize=(6,3.7));curve=curves[selection]
    for j,key in enumerate(['Accuracy','Balanced_accuracy','F1']):ax.plot(curve.threshold,curve[key],c=COLOURS[j],label=key.replace('_',' '))
    ax.axvline(t,c='.2',ls='--',label=f'Development cutoff {t:.3f}')
    ax.set(xlabel='Probability decision threshold',ylabel='Development OOF score',ylim=(0,1));ax.legend(fontsize=8)
    save_figure(fig,figdir,'10_training_threshold_selection')
    fig,axes=plt.subplots(1,2,figsize=(9,4.8))
    top=importance.sort_values('Importance').tail(12);axes[0].barh(top.Feature,top.Importance,color=COLOURS[0]);axes[0].set_xlabel('Impurity importance')
    top=perm.sort_values('Importance_mean');axes[1].barh(top.Feature,top.Importance_mean,xerr=top.Importance_sd,color=COLOURS[2],capsize=2);axes[1].set_xlabel('Decrease in holdout balanced accuracy')
    for ax in axes:ax.tick_params(labelsize=7)
    fig.tight_layout();save_figure(fig,figdir,'11_feature_importance')
    fig,axes=plt.subplots(1,2,figsize=(8,4.4),sharex=True,sharey=True)
    for ax,lab in zip(axes,['Observed','Predicted']):
        labels=ytest if lab=='Observed' else (p>=t).astype(int)
        for cls,marker,col in [(0,'o',COLOURS[1]),(1,'^',COLOURS[2])]:
            ax.scatter(test.loc[labels==cls,'HR'],test.loc[labels==cls,'N_prime'],s=30,marker=marker,c=col,label=['Stable','Unstable'][cls])
        plot_boundaries(ax,boundaries);mathews_axes(ax);ax.set(title=lab+' holdout',xlim=(5.4,9.1),ylim=(2,55))
    outside_legend(fig,axes,ncol=3,fontsize=8,bottom=0.10)
    save_figure(fig,figdir,'12_holdout_observed_predicted')
    fig,axes=plt.subplots(1,3,figsize=(10,4.15),sharex=True,sharey=True)
    for ax,(lab,yy) in zip(axes,[(selection,p>=t),('Chart logistic',preds['Chart_logistic']['calibrated']>=thresholds['Chart_logistic']),('Potvin chart',empirical)]):
        for correct,marker,col in [(True,'o',COLOURS[1]),(False,'x',COLOURS[2])]:
            q=(yy==ytest)==correct;ax.scatter(test.loc[q,'HR'],test.loc[q,'N_prime'],marker=marker,c=col,s=28,label='Correct' if correct else 'Error')
        plot_boundaries(ax,boundaries);mathews_axes(ax);ax.set(title=lab.replace('_',' '),xlim=(5.4,9.1),ylim=(2,55))
    outside_legend(fig,axes,ncol=3,fontsize=8,bottom=0.11)
    save_figure(fig,figdir,'13_same_holdout_error_maps')
    hr,nn=grids;H,N=np.meshgrid(hr,nn)
    fig,axes=plt.subplots(1,2,figsize=(9,4),sharex=True,sharey=True)
    for ax,s in zip(axes,['HW','FW']):
        ax.set_facecolor('#E8EDF2');im=ax.pcolormesh(H,N,maps[s],cmap='viridis',vmin=0,vmax=1,shading='auto',rasterized=True)
        for level in [.2,.5,.8]:
            if np.nanmin(maps[s])<level<np.nanmax(maps[s]):
                cs=ax.contour(H,N,maps[s],levels=[level],colors='white',linewidths=.8);ax.clabel(cs,fmt='%.1f',fontsize=8)
        q=train[train.Surface==s];ax.scatter(q.HR,q.N_prime,s=5,c='black',alpha=.4)
        mathews_axes(ax);ax.set(title=s+' OOF probability projection',xlim=(hr.min(),hr.max()),ylim=(nn.min(),nn.max()))
    fig.colorbar(im,ax=axes,label='Projected probability of ELOS at least 0.6 m',shrink=.75)
    save_figure(fig,figdir,'14_surface_probability_projections')
    fig,axes=plt.subplots(1,2,figsize=(9,4),sharex=True,sharey=True)
    for ax,s in zip(axes,['HW','FW']):
        q=train[train.Surface==s];ax.scatter(q.HR,q.N_prime,s=22,c=(q.ELOS_m>=.6),cmap='coolwarm',vmin=0,vmax=1,alpha=.7)
        unsupported=~np.isfinite(maps[s]);ax.contourf(H,N,unsupported,levels=[.5,1.5],colors=['#E8EDF2'],alpha=.8)
        mathews_axes(ax);ax.set(title=s+' observed training support',xlim=(hr.min(),hr.max()),ylim=(nn.min(),nn.max()))
    save_figure(fig,figdir,'15_training_support_masks')
    fig,ax=plt.subplots(figsize=(6.5,4.5));ax.set_facecolor('#E8EDF2')
    im=ax.pcolormesh(H,N,maps['maximum'],cmap='viridis',vmin=0,vmax=1,shading='auto',rasterized=True)
    plot_boundaries(ax,boundaries);mathews_axes(ax);ax.set(xlim=(hr.min(),hr.max()),ylim=(nn.min(),nn.max()))
    ax.legend(fontsize=7,loc='lower left');fig.colorbar(im,ax=ax,label='Maximum of HW and FW projections')
    save_figure(fig,figdir,'16_maximum_probability_envelope')
    fig,axes=plt.subplots(1,2,figsize=(8,3.7),sharex=True,sharey=True)
    for j,stage in enumerate(['raw','calibrated']):
        pp=preds[selection][stage];edges=np.quantile(pp,np.linspace(0,1,6));edges[0]=-1;edges[-1]=2
        ids=np.digitize(pp,edges[1:-1]);xs=[];ys=[];counts=[]
        for b in range(5):
            q=ids==b
            if q.any():xs.append(pp[q].mean());ys.append(ytest[q].mean());counts.append(q.sum())
        axes[j].plot([0,1],[0,1],'--',c='.5');axes[j].plot(xs,ys,'o-',c=COLOURS[j])
        for xx,yy,n in zip(xs,ys,counts):axes[j].annotate(str(n),(xx,yy),xytext=(4,4),textcoords='offset points',fontsize=8)
        axes[j].set(title=stage.capitalize()+' holdout probabilities',xlabel='Mean predicted probability',ylabel='Observed unstable fraction',xlim=(0,1),ylim=(0,1))
    fig.tight_layout();save_figure(fig,figdir,'17_probability_calibration')
    fig,axes=plt.subplots(2,2,figsize=(9,7),sharex=True,sharey=True)
    for i,s in enumerate(['HW','FW']):
        weight=float((train.loc[train.Surface==s,'ELOS_m']>=.6).mean())
        for j in range(2):
            val=maps[s] if j==0 else maps[s]*weight
            ax=axes[i,j];ax.set_facecolor('#E8EDF2');im=ax.pcolormesh(H,N,val,cmap='viridis',vmin=0,vmax=1,shading='auto',rasterized=True)
            mathews_axes(ax);ax.set(title=s+(' unweighted probability' if j==0 else f' legacy score p times {weight:.3f}'),xlim=(hr.min(),hr.max()),ylim=(nn.min(),nn.max()))
    fig.colorbar(im,ax=axes,label='Probability or legacy weighted score',shrink=.8)
    save_figure(fig,figdir,'S01_weighting_diagnostic')
    fig,ax=plt.subplots(figsize=(6,4))
    for j,n in enumerate(['Extra_Trees_regression','Random_Forest_regression']):
        ax.scatter(test.ELOS_m,regression[n],s=22,alpha=.65,c=COLOURS[j],label=n.replace('_',' '))
    ax.plot([0,4],[0,4],'--',c='.5');ax.set(xlabel='Observed holdout ELOS (m)',ylabel='Predicted ELOS (m)',xlim=(0,4),ylim=(0,4));ax.legend(fontsize=8)
    save_figure(fig,figdir,'S02_continuous_elos_regression')


In [ ]:
def main(argv=None):
    parser=argparse.ArgumentParser(description='Grouped stope stability analysis')
    parser.add_argument('--data-dir',type=Path,default=Path('data'))
    parser.add_argument('--output-dir',type=Path,default=Path('results'))
    args=parser.parse_args(argv);out=args.output_dir;out.mkdir(parents=True,exist_ok=True)
    tables=out/'tables';tables.mkdir(exist_ok=True)
    raw=pd.read_excel(args.data_dir/'STOPE ID.xlsx',sheet_name='Stope Data')
    raw.columns=[str(c).strip() for c in raw.columns];raw=raw.rename(columns=RENAME)
    raw['source_excel_row']=np.arange(len(raw))+2
    raw['stope_id']=raw['Stope Name'].astype(str).str.strip()
    assert raw[['stope_id','Surface']].duplicated().sum()==0
    assert raw.groupby('stope_id').Surface.apply(set).apply(lambda s:s=={'HW','FW'}).all()
    reason=np.where(raw.Q_prime.isna() & (raw.N_prime<=0),"Missing Q_prime and nonpositive N_prime",'')
    invalid=(raw.N_prime<=0)|raw.ELOS_m.isna()|raw.stope_id.eq('nan')
    if np.any(invalid & (reason=='')):raise ValueError('Unclassified invalid record requires review')

    raw['analysis_status']=[f'Excluded: {r}' if bad else 'Included'
                            for bad,r in zip(invalid,reason)]
    raw.to_csv(tables/'source_record_audit.csv',index=False)
    df=raw.loc[~invalid].reset_index(drop=True)
    assert df.groupby('stope_id').size().eq(2).all()
    X=df[BASE_NUM+['Surface']].copy();y=(df.ELOS_m>=.6).to_numpy(dtype=int);g=df.stope_id
    a,b=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED).split(X,y,g))
    assert not set(g.iloc[a]) & set(g.iloc[b])
    train=df.iloc[a].reset_index(drop=True);test=df.iloc[b].reset_index(drop=True)
    Xt=X.iloc[a].reset_index(drop=True);Xv=X.iloc[b].reset_index(drop=True)
    yt=y[a];yv=y[b];gt=g.iloc[a].reset_index(drop=True);gv=g.iloc[b].reset_index(drop=True)
    split=df[['source_excel_row','stope_id','Surface','ELOS_m']].copy();split['subset']='development';split.loc[b,'subset']='holdout'
    split.to_csv(tables/'split_manifest.csv',index=False)
    df.select_dtypes(include='number').drop(columns='source_excel_row').describe().T.to_csv(tables/'descriptive_statistics.csv')
    boundaries=boundary_load(args.data_dir/'POTVIN BOUNDARIES.xlsx')
    cv=splits(Xt,yt,gt)
    templates={
        'Chart_logistic':pipeline(LogisticRegression(C=1e6,max_iter=6000),Xt,kind='simple'),
        'Full_logistic':pipeline(LogisticRegression(C=1,max_iter=6000,class_weight='balanced',random_state=SEED),Xt),
        'Random_Forest':pipeline(RandomForestClassifier(n_estimators=1000,max_depth=12,min_samples_leaf=2,class_weight='balanced',random_state=SEED,n_jobs=1),Xt),
        'Extra_Trees_fixed':pipeline(ExtraTreesClassifier(n_estimators=1000,max_depth=12,min_samples_leaf=2,class_weight='balanced',random_state=SEED,n_jobs=1),Xt),
        'Extra_Trees_tuned':pipeline(ExtraTreesClassifier(class_weight='balanced',random_state=SEED,n_jobs=1),Xt)}
    models={};dev={};thresholds={};calibrators={};curves={};rows=[];folds=[];searches={}
    for name,template in templates.items():
        emit('Development grouped CV: '+name)
        po,fr,settings=raw_oof(template,Xt,yt,gt,cv,tune=name=='Extra_Trees_tuned')
        for row in fr:row['Model']=name
        folds.extend(fr);frame=pd.DataFrame(fr)
        rows.append({'Model':name,**{metric+'_mean':float(frame[metric].mean()) for metric in METRICS},
                              **{metric+'_sd':float(frame[metric].std(ddof=1)) for metric in METRICS}})
        pc,theta=cross_calibration(yt,po,cv);th,curve=choose_threshold(yt,pc)
        thresholds[name]=th;curves[name]=curve;calibrators[name]=theta
        dev[name]={'raw':po,'calibrated_crossfit':pc}
        if name=='Extra_Trees_tuned':
            model,params,score=search_fit(template,Xt,yt,gt)
            searches[name]={'outer_fold_searches':settings,'final_parameters':params,'final_inner_score':score}
        else:model=clone(template).fit(Xt,yt)
        models[name]=model
    cvframe=pd.DataFrame(rows);foldframe=pd.DataFrame(folds)

    selection=cvframe.sort_values(['Balanced_accuracy_mean','ROC_AUC_mean'],ascending=False,kind='stable').iloc[0].Model
    emit('Frozen development selection: '+selection)
    cvframe.to_csv(tables/'grouped_cv_summary.csv',index=False);foldframe.to_csv(tables/'grouped_cv_folds.csv',index=False)
    audit={'source_records':len(raw),'source_stopes':raw.stope_id.nunique(),
        'excluded_records':int(invalid.sum()),'valid_records':len(df),'valid_stopes':df.stope_id.nunique(),
        'train_records':len(train),'train_stopes':gt.nunique(),'test_records':len(test),'test_stopes':gv.nunique(),
        'train_unstable':int(yt.sum()),'test_unstable':int(yv.sum()),'selected_model':selection,
        'selection_rule':'Mean five-fold grouped development balanced accuracy at raw probability threshold 0.5; ROC AUC breaks ties',
        'thresholds':thresholds,'calibrators':{k:v.tolist() for k,v in calibrators.items()},'searches':searches,
        'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'scikit_learn':sklearn.__version__,'matplotlib':matplotlib.__version__},
        'input_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [args.data_dir/'STOPE ID.xlsx',args.data_dir/'POTVIN BOUNDARIES.xlsx']}}
    (out/'frozen_development_protocol.json').write_text(json.dumps(audit,indent=2))
    resamples=cluster_indices(gv);preds={};holdout=[];calibration=[]
    export=test[['source_excel_row','stope_id','Surface','HR','N_prime','ELOS_m']].copy();export['observed_unstable']=yv
    for name,model in models.items():
        emit('Locked holdout evaluation: '+name)
        po=model.predict_proba(Xv)[:,1];pc=sigmoid_apply(po,calibrators[name]);preds[name]={'raw':po,'calibrated':pc}
        vals=metric_values(yv,pc,thresholds[name]);ci=metric_intervals(yv,pc,thresholds[name],resamples)
        holdout.append({'Model':name,'Threshold':thresholds[name],**vals,
                       **{key+'_CI_low':val[0] for key,val in ci.items()},**{key+'_CI_high':val[1] for key,val in ci.items()}})
        for stage,pr in [('raw',po),('calibrated',pc)]:calibration.append({'Model':name,'Stage':stage,**calibration_diagnostics(yv,pr)})
        export[name+'_probability']=pc;export[name+'_predicted']=(pc>=thresholds[name]).astype(int)
        curves[name].to_csv(tables/(name+'_development_threshold_curve.csv'),index=False)
    for name in ['Stable','Stable-Unstable']:
        pe,limit=boundary_prediction(test,boundaries,name)
        label='Potvin_'+name.replace('-','_');preds[label]={'raw':pe.astype(float),'calibrated':pe.astype(float)};thresholds[label]=.5
        ci=metric_intervals(yv,pe,.5,resamples);vals=metric_values(yv,pe,.5)

        for key in ['ROC_AUC','AP','Brier']:vals[key]=np.nan;ci[key]=(np.nan,np.nan)
        holdout.append({'Model':label,'Threshold':.5,**vals,
            **{key+'_CI_low':val[0] for key,val in ci.items()},**{key+'_CI_high':val[1] for key,val in ci.items()}})
        export[label+'_predicted']=pe;export[label+'_boundary']=limit
    holdout=pd.DataFrame(holdout);holdout.to_csv(tables/'holdout_results.csv',index=False)
    calibration=pd.DataFrame(calibration);calibration.to_csv(tables/'calibration_diagnostics.csv',index=False)
    export.to_csv(tables/'holdout_predictions.csv',index=False)
    comparisons=[]
    for n1,n2 in [(selection,'Chart_logistic'),(selection,'Random_Forest'),(selection,'Potvin_Stable'),('Extra_Trees_tuned','Extra_Trees_fixed')]:
        if n1==n2:continue
        comparisons.append({'Model_1':n1,'Model_2':n2,**paired_compare(yv,preds[n1]['calibrated'],thresholds[n1],preds[n2]['calibrated'],thresholds[n2],gv,resamples)})
    pd.DataFrame(comparisons).to_csv(tables/'paired_model_comparisons.csv',index=False)

    tree_name=selection if 'Trees' in selection or 'Forest' in selection else 'Extra_Trees_tuned'
    tree_model=models[tree_name].named_steps['model']
    ablation=[]
    for name,surface,interaction,kind in [('Full_features',True,True,'full'),('Without_Surface',False,True,'full'),('Without_interaction',True,False,'full'),('Raw_features_only',True,False,'raw')]:
        emit('Grouped feature ablation: '+name)
        template=pipeline(clone(tree_model),Xt,kind=kind,surface=surface,interaction=interaction)
        po,fr,_=raw_oof(template,Xt,yt,gt,cv);pc,theta=cross_calibration(yt,po,cv);th,_=choose_threshold(yt,pc)
        model=clone(template).fit(Xt,yt);pv=sigmoid_apply(model.predict_proba(Xv)[:,1],theta)
        ablation.append({'Variant':name,'Model':tree_name,'Threshold':th,
            'CV_Balanced_accuracy_mean':float(np.mean([r['Balanced_accuracy'] for r in fr])),
            'CV_Balanced_accuracy_sd':float(np.std([r['Balanced_accuracy'] for r in fr],ddof=1)),**metric_values(yv,pv,th)})
    pd.DataFrame(ablation).to_csv(tables/'feature_ablations.csv',index=False)
    sensitivity=[]
    for cutoff in [.5,.6,.8]:
        emit('ELOS cutoff sensitivity: '+str(cutoff))
        ys=(train.ELOS_m>=cutoff).to_numpy(dtype=int);yv_s=(test.ELOS_m>=cutoff).to_numpy(dtype=int)
        cv_s=splits(Xt,ys,gt);template=clone(models[selection])
        po,fr,_=raw_oof(template,Xt,ys,gt,cv_s);pc,theta=cross_calibration(ys,po,cv_s);th,_=choose_threshold(ys,pc)
        fitted=clone(template).fit(Xt,ys);pv=sigmoid_apply(fitted.predict_proba(Xv)[:,1],theta)
        sensitivity.append({'ELOS_cutoff_m':cutoff,'Train_unstable':int(ys.sum()),'Test_unstable':int(yv_s.sum()),
            'HW_unstable_total':int(((df.Surface=='HW') & (df.ELOS_m>=cutoff)).sum()),
            'FW_unstable_total':int(((df.Surface=='FW') & (df.ELOS_m>=cutoff)).sum()),
            'Threshold':th,**metric_values(yv_s,pv,th)})
    pd.DataFrame(sensitivity).to_csv(tables/'elos_cutoff_sensitivity.csv',index=False)

    regression={};reg_rows=[]
    for name,estimator in [('Extra_Trees_regression',ExtraTreesRegressor(n_estimators=600,max_depth=12,min_samples_leaf=2,random_state=SEED,n_jobs=1)),
                           ('Random_Forest_regression',RandomForestRegressor(n_estimators=600,max_depth=12,min_samples_leaf=2,random_state=SEED,n_jobs=1))]:
        emit('Grouped continuous ELOS regression: '+name)
        template=pipeline(estimator,Xt);oof=np.zeros(len(Xt))
        for aa,bb in cv:oof[bb]=clone(template).fit(Xt.iloc[aa],train.ELOS_m.iloc[aa]).predict(Xt.iloc[bb])
        fitted=clone(template).fit(Xt,train.ELOS_m);pv=fitted.predict(Xv);regression[name]=pv
        assert (pv>=0).all()
        stats={'CV_MAE_m':float(mean_absolute_error(train.ELOS_m,oof)),
            'Test_MAE_m':float(mean_absolute_error(test.ELOS_m,pv)),
            'Test_RMSE_m':float(root_mean_squared_error(test.ELOS_m,pv)),
            'Test_R2':float(r2_score(test.ELOS_m,pv))}
        boot=np.asarray([[mean_absolute_error(test.ELOS_m.iloc[ix],pv[ix]),root_mean_squared_error(test.ELOS_m.iloc[ix],pv[ix]),r2_score(test.ELOS_m.iloc[ix],pv[ix])] for ix in resamples])
        for j,key in enumerate(['Test_MAE_m','Test_RMSE_m','Test_R2']):stats[key+'_CI_low'],stats[key+'_CI_high']=np.quantile(boot[:,j],[.025,.975])
        reg_rows.append({'Model':name,**stats})
    pd.DataFrame(reg_rows).to_csv(tables/'continuous_elos_regression.csv',index=False)
    pd.DataFrame(regression).assign(source_excel_row=test.source_excel_row).to_csv(tables/'continuous_elos_predictions.csv',index=False)
    chosen=models[tree_name]
    pi=permutation_importance(chosen,Xv,yv,scoring='balanced_accuracy',n_repeats=50,random_state=SEED,n_jobs=4)
    perm=pd.DataFrame({'Feature':Xv.columns,'Importance_mean':pi.importances_mean,'Importance_sd':pi.importances_std})
    importance=pd.DataFrame({'Feature':chosen.named_steps['preprocess'].get_feature_names_out(),
                              'Importance':chosen.named_steps['model'].feature_importances_})
    perm.to_csv(tables/'permutation_importance.csv',index=False);importance.to_csv(tables/'impurity_importance.csv',index=False)
    oof_frame=train[['source_excel_row','stope_id','Surface','HR','N_prime','ELOS_m']].copy()
    for name in dev:oof_frame[name+'_raw']=dev[name]['raw'];oof_frame[name+'_calibrated_crossfit']=dev[name]['calibrated_crossfit']
    oof_frame.to_csv(tables/'development_oof_predictions.csv',index=False)
    grid_hr=np.linspace(train.HR.min(),train.HR.max(),150);grid_n=np.geomspace(train.N_prime.min(),train.N_prime.max(),150)
    maps={};support={};radii={};distances={}
    for s in ['HW','FW']:
        mask=train.Surface==s
        maps[s],support[s],radii[s],distances[s]=supported_projection(train[mask],dev[selection]['calibrated_crossfit'][mask],grid_hr,grid_n)
    maps['maximum']=np.maximum(maps['HW'],maps['FW'])
    H,N=np.meshgrid(grid_hr,grid_n)
    pd.DataFrame({'HR':H.ravel(),'N_prime':N.ravel(),'HW_probability':maps['HW'].ravel(),
        'FW_probability':maps['FW'].ravel(),'Maximum_probability':maps['maximum'].ravel(),
        'HW_supported':support['HW'].ravel(),'FW_supported':support['FW'].ravel(),
        'HW_fifth_neighbor_distance':distances['HW'].ravel(),'FW_fifth_neighbor_distance':distances['FW'].ravel()}).to_csv(tables/'supported_probability_grid.csv',index=False)
    audit['support_radius_95_percentile']=radii
    audit['training_surface_unstable_fractions']={s:float(yt[train.Surface==s].mean()) for s in ['HW','FW']}
    audit['confusion_matrix']=confusion_matrix(yv,preds[selection]['calibrated']>=thresholds[selection]).tolist()
    audit['feature_count']=len(features(Xt).columns)
    audit['empirical_all_supplied_valid_accuracy']=float(accuracy_score(y,boundary_prediction(df,boundaries)[0]))
    audit['regression_development_selection']=min(reg_rows,key=lambda r:r['CV_MAE_m'])['Model']
    audit['test_HR_N_error_coordinates']={n:export.loc[export[n+'_predicted']!=yv,['source_excel_row','HR','N_prime']].to_dict('records') for n in [selection,'Chart_logistic','Potvin_Stable']}
    (out/'analysis_summary.json').write_text(json.dumps(audit,indent=2))
    make_figures(df,train,test,yt,yv,models,selection,cvframe,foldframe,preds,thresholds,curves,
                 perm,importance,boundaries,maps,(grid_hr,grid_n),preds['Potvin_Stable']['calibrated'].astype(int),ablation,sensitivity,regression,out)
    emit('Final holdout metrics:\n'+holdout[['Model','Accuracy','Balanced_accuracy','F1','ROC_AUC','AP','Brier']].to_string(index=False))
    emit('Analysis complete. Results: '+str(out.resolve()))
    postchecks(out)
    return audit


In [ ]:
DATA_DIR = Path("data")
RESULTS_DIR = Path("reproduced_results")
assert (DATA_DIR / "STOPE ID.xlsx").is_file()
assert (DATA_DIR / "POTVIN BOUNDARIES.xlsx").is_file()
analysis_summary = main(["--data-dir", str(DATA_DIR), "--output-dir", str(RESULTS_DIR)])


Development grouped CV: Chart_logistic
Development grouped CV: Full_logistic
Development grouped CV: Random_Forest
Development grouped CV: Extra_Trees_fixed
Development grouped CV: Extra_Trees_tuned
Frozen development selection: Extra_Trees_tuned
Locked holdout evaluation: Chart_logistic
Locked holdout evaluation: Full_logistic
Locked holdout evaluation: Random_Forest
Locked holdout evaluation: Extra_Trees_fixed
Locked holdout evaluation: Extra_Trees_tuned
Grouped feature ablation: Full_features
Grouped feature ablation: Without_Surface
Grouped feature ablation: Without_interaction
Grouped feature ablation: Raw_features_only
ELOS cutoff sensitivity: 0.5
ELOS cutoff sensitivity: 0.6
ELOS cutoff sensitivity: 0.8
Grouped continuous ELOS regression: Extra_Trees_regression
Grouped continuous ELOS regression: Random_Forest_regression


'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very l

Final holdout metrics:
                 Model  Accuracy  Balanced_accuracy       F1  ROC_AUC       AP    Brier
        Chart_logistic  0.770270           0.742105 0.604651 0.796172 0.602376 0.155498
         Full_logistic  0.797297           0.760287 0.634146 0.815311 0.621904 0.171598
         Random_Forest  0.770270           0.776555 0.638298 0.753110 0.557192 0.160896
     Extra_Trees_fixed  0.837838           0.804785 0.700000 0.774163 0.580937 0.159877
     Extra_Trees_tuned  0.797297           0.725837 0.594595 0.760766 0.550745 0.155972
         Potvin_Stable  0.567568           0.657416 0.500000      NaN      NaN      NaN
Potvin_Stable_Unstable  0.729730           0.525359 0.166667      NaN      NaN      NaN
Analysis complete. Results: C:\Users\Onalethata\Desktop\Kefilwe\reproduced_results
Numerical reconciliation passed; calibration cluster intervals exported.


In [ ]:
pd.read_csv(RESULTS_DIR / "tables" / "holdout_results.csv")[
    ["Model", "Threshold", "Accuracy", "Balanced_accuracy", "F1", "ROC_AUC", "AP", "Brier"]
]


,Model,Threshold,Accuracy,Balanced_accuracy,F1,ROC_AUC,AP,Brier
0,Chart_logistic,0.370,0.770270,0.742105,0.604651,0.796172,0.602376,0.155498
1,Full_logistic,0.335,0.797297,0.760287,0.634146,0.815311,0.621904,0.171598
2,Random_Forest,0.310,0.770270,0.776555,0.638298,0.753110,0.557192,0.160896
3,Extra_Trees_fixed,0.355,0.837838,0.804785,0.700000,0.774163,0.580937,0.159877
4,Extra_Trees_tuned,0.355,0.797297,0.725837,0.594595,0.760766,0.550745,0.155972
5,Potvin_Stable,0.500,0.567568,0.657416,0.500000,NaN,NaN,NaN
6,Potvin_Stable_Unstable,0.500,0.729730,0.525359,0.166667,NaN,NaN,NaN


In [ ]:
pd.read_csv(RESULTS_DIR / "tables" / "feature_ablations.csv")


,Variant,Model,Threshold,CV_Balanced_accuracy_mean,CV_Balanced_accuracy_sd,Accuracy,Balanced_accuracy,F1,MCC,ROC_AUC,AP,Brier,Precision,Recall
0,Full_features,Extra_Trees_tuned,0.360,0.671717,0.098687,0.797297,0.725837,0.594595,0.459890,0.760766,0.550745,0.157755,0.611111,0.578947
1,Without_Surface,Extra_Trees_tuned,0.305,0.667685,0.125308,0.756757,0.715789,0.571429,0.407352,0.762679,0.559954,0.160062,0.521739,0.631579
2,Without_interaction,Extra_Trees_tuned,0.395,0.671270,0.087921,0.797297,0.674163,0.516129,0.412816,0.768421,0.591064,0.156872,0.666667,0.421053
3,Raw_features_only,Extra_Trees_tuned,0.365,0.678413,0.097345,0.824324,0.778469,0.666667,0.547839,0.774163,0.611058,0.156528,0.650000,0.684211
